# New Analyst Workshop: Learner Engagement and Quiz Outcomes

## What we're analyzing

This lesson turns one workshop snapshot into a trustworthy beginner analysis bundle. We will use `lesson_brief.md` for the teaching goal, `learner_events.csv` for session progress, `quiz_attempts.csv` for assessment outcomes, and `quiz_items.csv` for topic-level misconception signals.

Source anchors: `lesson_brief.md`, `learner_events.csv`, `quiz_attempts.csv`, `quiz_items.csv`.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import yaml
import matplotlib.pyplot as plt

plt.switch_backend('Agg')
WORKSPACE = Path(os.environ.get('WORKSPACE_ROOT', '/app/workspace'))
learner_events = pd.read_csv(WORKSPACE / 'learner_events.csv')
quiz_attempts = pd.read_csv(WORKSPACE / 'quiz_attempts.csv')
quiz_items = pd.read_csv(WORKSPACE / 'quiz_items.csv')
metric_definitions = yaml.safe_load((WORKSPACE / 'metric_definitions.yaml').read_text(encoding='utf-8'))
lesson_brief = (WORKSPACE / 'lesson_brief.md').read_text(encoding='utf-8')
print('Rows in learner_events.csv:', len(learner_events))
print('Rows in quiz_attempts.csv:', len(quiz_attempts))
print('Rows in quiz_items.csv:', len(quiz_items))
print('Metrics defined:', [m['name'] for m in metric_definitions['metrics']])

## Understand the event data

Start with the event log before making claims. `learner_events.csv` tells us who started the workshop, who completed the lesson, and who opened or submitted practice. `metric_definitions.yaml` is the source of truth for learner-level rates. `reference_docs/glossary.md` defines the event names in plain language.

In [ ]:
event_profile = (learner_events.groupby('event_name')['learner_id']
    .nunique()
    .rename('unique_learners')
    .reset_index()
    .sort_values('unique_learners', ascending=False))
event_profile

## Build the session funnel

The funnel should use unique learners, not raw event rows. That rule comes directly from `metric_definitions.yaml` and is reinforced by `reference_docs/facilitation_notes.md`.

In [ ]:
funnel_order = ['session_started', 'lesson_completed', 'practice_opened', 'practice_submitted']
funnel = pd.DataFrame({
    'step': funnel_order,
    'unique_learners': [
        learner_events.loc[learner_events['event_name'] == step, 'learner_id'].nunique()
        for step in funnel_order
    ],
})
funnel['share_of_starters'] = (funnel['unique_learners'] / funnel.loc[funnel['step'] == 'session_started', 'unique_learners'].iloc[0]).round(4)
funnel

In [ ]:
ax = funnel.plot(x='step', y='unique_learners', kind='bar', legend=False, title='Workshop funnel')
ax.set_ylabel('Unique learners')
plt.tight_layout()
plt.show()

## Compare quiz outcomes

Quiz metrics need learner-level rollups. `quiz_attempts.csv` gives attempt-level rows, but `quiz_pass_rate` in `metric_definitions.yaml` is defined on unique learners with any passing attempt. We can also use `quiz_items.csv` to surface which topics still create confusion for learners.

In [ ]:
quiz_by_learner = (quiz_attempts.sort_values(['learner_id', 'attempt_number'])
    .groupby('learner_id')
    .agg(
        attempts=('attempt_number', 'max'),
        first_score=('score_pct', 'first'),
        best_score=('score_pct', 'max'),
        passed_any=('passed', 'max'),
    )
    .reset_index())
quiz_by_learner

In [ ]:
quiz_summary = pd.DataFrame({
    'metric': ['quiz_pass_rate', 'retry_rate', 'median_first_score'],
    'value': [
        round(quiz_by_learner['passed_any'].mean(), 4),
        round((quiz_by_learner['attempts'] > 1).mean(), 4),
        round(quiz_by_learner['first_score'].median(), 2),
    ],
})
quiz_summary

In [ ]:
misconception_summary = quiz_items.assign(error_rate=(quiz_items['incorrect_learners'] / (quiz_items['correct_learners'] + quiz_items['incorrect_learners'])).round(4)).sort_values('error_rate', ascending=False)
misconception_summary[['question_id', 'topic', 'error_rate', 'common_misconception']]

## Spot metric definition traps

Two easy mistakes in this task are called out in the visible inputs. `metric_definitions.yaml` warns against counting raw events for `completion_rate`, `reference_docs/glossary.md` clarifies that `retry_rate` is about retrying learners, not retrying attempts, and `quiz_items.csv` helps show where learners misread the metric logic.

In [ ]:
started = learner_events.loc[learner_events['event_name'] == 'session_started', 'learner_id'].nunique()
completed = learner_events.loc[learner_events['event_name'] == 'lesson_completed', 'learner_id'].nunique()
practice_opened = learner_events.loc[learner_events['event_name'] == 'practice_opened', 'learner_id'].nunique()
practice_submitted = learner_events.loc[learner_events['event_name'] == 'practice_submitted', 'learner_id'].nunique()
attempted = quiz_attempts['learner_id'].nunique()
retrying_learners = quiz_attempts.loc[quiz_attempts['attempt_number'] > 1, 'learner_id'].nunique()
passed_any = quiz_attempts.loc[quiz_attempts['passed'] == 1, 'learner_id'].nunique()

metric_table = pd.DataFrame({
    'metric_name': ['completion_rate', 'practice_submission_rate', 'quiz_pass_rate', 'retry_rate'],
    'value': [
        round(completed / started, 4),
        round(practice_submitted / practice_opened, 4),
        round(passed_any / attempted, 4),
        round(retrying_learners / attempted, 4),
    ],
    'definition_source': ['metric_definitions.yaml'] * 4,
})
metric_table

## Practice

Q1. Using `learner_events.csv`, which funnel step loses the most learners relative to the previous step, and how would you explain that to a new stakeholder?

Q2. Why would attempt-level averaging understate or overstate `quiz_pass_rate` compared with the learner-level definition in `metric_definitions.yaml`?

Q3. Looking at `quiz_items.csv`, which misconception would you prioritize in the next workshop and why?

## Wrap up

This workshop snapshot shows a healthy start rate but visible attrition between lesson completion and practice submission. It also shows why metric definitions matter: `completion_rate`, `practice_submission_rate`, `quiz_pass_rate`, and `retry_rate` all need learner-level logic from `metric_definitions.yaml`, while `quiz_items.csv` reveals that metric-definition topics still produce learner confusion.

Sources used in this notebook: `lesson_brief.md`, `learner_events.csv`, `quiz_attempts.csv`, `quiz_items.csv`, `metric_definitions.yaml`, `reference_docs/glossary.md`, `reference_docs/facilitation_notes.md`.